<a href="https://colab.research.google.com/github/Christi0-0R/sbox-ga/blob/fase-5-corridas-finales/notebooks/sbox_ga_verificacion_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# sbox-ga · Verificación en Colab (Fase 5)

Notebook ordenado para **reproducir y verificar** los resultados finales con sboxU.
Se ejecuta **de arriba hacia abajo**; cada celda dice qué espera ver al terminar.

| Paso | Qué hace | Tiempo aprox. |
|---|---|---|
| 0 | Parámetros (lo único que se podría editar) | — |
| 1 | Monta Drive y clona la rama | < 1 min |
| 2 | Instala Sage + sboxU (se salta si ya está) | 15–25 min la primera vez |
| 3 | Pruebas + contraste de las S-boxes con sboxU | ~5 min |
| 4 | Reproducibilidad: recalcula 3 semillas y compara byte a byte con tu laptop | ~5–10 min |
| 5 | (opcional) Regenera el análisis estadístico | 1 min |
| 6 | Guarda la evidencia en Drive y la descarga | < 1 min |

**Requisitos previos:** la rama y el tag están subidos a GitHub, y `resultados/final/` está en el repo (o respaldado en Drive).
**No se re-corren las 300 corridas:** ya están hechas; aquí solo se verifican.

## 0 · Parámetros

In [1]:
import os

REPO_URL  = "https://github.com/Christi0-0R/sbox-ga.git"
RAMA      = "fase-5-corridas-finales"      # o "congelado-v1" para el código exacto del tag
DIR       = "/content/sbox-ga"
SAGE      = "/content/sage-env"            # entorno aparte con Sage y sboxU
DRIVE_DIR = "/content/drive/MyDrive/Sbox-Ga"

EJECUTAR_REPRO    = True    # paso 4: recalcular 3 semillas y comparar con resultados/final
SEMILLAS_REPRO    = "1000,1001,1002"
EJECUTAR_ANALISIS = False   # paso 5: regenerar tablas y gráficas

# las celdas %%bash leen estas variables
os.environ.update(REPO_URL=REPO_URL, RAMA=RAMA, DIR=DIR, SAGE=SAGE, DRIVE_DIR=DRIVE_DIR,
                  SEMILLAS_REPRO=SEMILLAS_REPRO)
print("Parámetros listos")
os.environ['EJECUTAR_REPRO'] = '1' if EJECUTAR_REPRO else '0'
os.environ['EJECUTAR_ANALISIS'] = '1' if EJECUTAR_ANALISIS else '0'


Parámetros listos


## 1 · Drive y repositorio
Esperado: lista de 3 commits, el tag/commit actual y `200` S-boxes.
(Se hace `cd /content` antes de borrar para evitar el error `getcwd`.)

In [2]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

os.chdir("/content")
!rm -rf {DIR}
!git clone -q {REPO_URL} {DIR}
os.chdir(DIR)
!git checkout -q {RAMA}
!git log --oneline -3
!git describe --tags --always

# si resultados/final no vino en el repo, se toma del respaldo en Drive
if not os.path.isdir(f"{DIR}/resultados/final/sboxes"):
    !mkdir -p {DIR}/resultados && cp -r {DRIVE_DIR}/final {DIR}/resultados/
n = len(os.listdir(f"{DIR}/resultados/final/sboxes"))
print("S-boxes encontradas:", n)
assert n == 200, f"se esperaban 200 S-boxes y hay {n}"

Mounted at /content/drive
b27f11f (HEAD -> fase-5-corridas-finales, origin/fase-5-corridas-finales) Correccion del README
96c635b Demo y notebook
2d47360 demo
congelado-v1-5-gb27f11f
S-boxes encontradas: 200


## 2 · Sage y sboxU
Se instalan en un entorno aparte (`/content/sage-env`, con su propio Python 3.14).
Cada celda **se salta sola si ya está instalado** y **se detiene con el error** si algo falla.

**2a · Sage** (la primera vez tarda 10–20 min; no cierres la pestaña). Esperado: `sage OK`.

In [3]:
%%bash
set -euo pipefail
export PATH="$SAGE/bin:$PATH"
if [ ! -x "$SAGE/bin/python" ]; then
  echo ">> Instalando micromamba y Sage (10-20 min)"
  cd /content
  curl -fL --retry 5 --retry-delay 10 https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
  export MAMBA_ROOT_PREFIX=/content/mamba
  ./bin/micromamba create -y -q -p "$SAGE" -c conda-forge sage c-compiler cxx-compiler libgomp \
      > /content/mamba_create.log 2>&1 || { tail -20 /content/mamba_create.log; exit 1; }
else
  echo ">> El entorno de Sage ya existe, se reutiliza"
fi
python -c "import sage.all; print('sage OK')"

>> Instalando micromamba y Sage (10-20 min)
sage OK


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 7146k  100 7146k    0     0  2893k      0  0:00:02  0:00:02 --:--:--     0


**2b · sboxU y dependencias del proyecto** (compilar tarda 3–10 min). Esperado: `sboxU OK`.

In [4]:
%%bash
set -euo pipefail
export PATH="$SAGE/bin:$PATH"
if python -c "import sboxU" 2>/dev/null; then
  echo ">> sboxU ya está instalado"
else
  which g++ >/dev/null || apt-get install -y -q g++ libomp-dev
  python -m pip install -q setuptools wheel "cython>=3.0" pybtex rich sphinx-rtd-theme
  python -m pip install --no-build-isolation git+https://github.com/lpp-crypto/sboxU \
      > /content/sboxu_install.log 2>&1 || { tail -25 /content/sboxu_install.log; exit 1; }
fi
python -c "from sboxU import linearity, differential_uniformity; print('sboxU OK')"
python -m pip install -q -r "$DIR/requirements.txt"
python --version

sboxU OK
Python 3.14.10


## 3 · Pruebas y contraste con sboxU
Corre la suite completa, el contraste de 51 S-boxes (AES + 50 aleatorias) y el de las **200 S-boxes finales**.
Esperado:
- `104 passed` (sin saltos: `test_sboxu.py` ya no se salta),
- `51/51 coinciden (con sboxU)`,
- `200/200 S-boxes coinciden (con sboxU)`.

Si el contraste corre **sin** sboxU, la celda se detiene con error (así no se da por bueno un contraste parcial).
Los registros quedan en `/content/evidencia/`.

In [5]:
%%bash
set -euo pipefail
export PATH="$SAGE/bin:$PATH"
cd "$DIR"
mkdir -p /content/evidencia
{
  date
  python --version
  python -m pip list 2>/dev/null | grep -i -E "^(sboxu|numpy|scipy|pandas|deap|pytest) "
  echo "commit: $(git rev-parse HEAD)"
  echo "ref:    $(git describe --tags --always)"
} > /content/evidencia/entorno_colab.txt

python -m pytest -q 2>&1 | tee /content/evidencia/pytest.txt
python -m experimentos.contraste_sboxu 2>&1 | tee /content/evidencia/contraste_sboxu_51.txt
python -m experimentos.contraste_final resultados/final 2>&1 | tee /content/evidencia/contraste_final_200.txt

grep -q "con sboxU" /content/evidencia/contraste_sboxu_51.txt  || { echo "ERROR: contraste de 51 S-boxes SIN sboxU"; exit 1; }
grep -q "con sboxU" /content/evidencia/contraste_final_200.txt || { echo "ERROR: contraste final SIN sboxU"; exit 1; }
echo ">> Verificación completa: todo coincide con sboxU"

........................................................................ [ 69%]
................................                                         [100%]
104 passed in 173.90s (0:02:53)
51/51 coinciden (con sboxU)
200/200 S-boxes coinciden (con sboxU)
>> Verificación completa: todo coincide con sboxU


## 4 · Reproducibilidad entre máquinas
Recalcula en Colab unas pocas semillas (estática, adaptativa y aleatoria) con la configuración congelada
y compara **byte a byte** los CSV y las S-boxes contra `resultados/final` (generado en tu laptop).
Esperado: `IDENTICOS` en todo. Si hay diferencias, se muestra la primera línea distinta;
suele deberse a la versión de Python o de las librerías: no invalida los resultados, pero se anota en la bitácora.

In [7]:
%%bash
export PATH="$SAGE/bin:$PATH"
cd "$DIR"
if [ "${EJECUTAR_REPRO:-1}" = "0" ]; then echo "Paso 4 omitido (EJECUTAR_REPRO=False)"; exit 0; fi
python - <<'PYEND' 2>&1 | tee /content/evidencia/reproducibilidad.txt
import os, sys, tempfile
sys.path.insert(0, os.getcwd())
from experimentos import correr_todo as ct

semillas = [int(s) for s in os.environ["SEMILLAS_REPRO"].split(",")]
tmp = tempfile.mkdtemp(prefix="repro_")
iguales, distintas = 0, []
for v in ct.VARIANTES:
    for s in semillas:
        ct._tarea((v, s, tmp, dict(ct.CFG_CONGELADA)))
        pares = [(ct.ruta_csv(tmp, v, s), ct.ruta_csv("resultados/final", v, s))]
        if v != "aleatoria":
            pares.append((ct.ruta_sbox(tmp, v, s), ct.ruta_sbox("resultados/final", v, s)))
        for nuevo, final in pares:
            a, b = open(nuevo, "rb").read(), open(final, "rb").read()
            if a == b:
                iguales += 1
            else:
                la, lb = a.decode().splitlines(), b.decode().splitlines()
                k = next((i for i, (x, y) in enumerate(zip(la, lb)) if x != y), min(len(la), len(lb)))
                distintas.append((os.path.basename(final), k + 1))
total = iguales + len(distintas)
print(f"{iguales}/{total} archivos IDENTICOS a resultados/final" if not distintas
      else f"{iguales}/{total} identicos; DIFERENTES: {distintas}")
sys.exit(0 if not distintas else 1)
PYEND

15/15 archivos IDENTICOS a resultados/final


## 5 · (Opcional) Regenerar el análisis
Pon `EJECUTAR_ANALISIS = True` en el paso 0. Es determinista: debe dar las mismas tablas que en la laptop.

In [ ]:
%%bash
export PATH="$SAGE/bin:$PATH"
cd "$DIR"
if [ "${EJECUTAR_ANALISIS:-0}" != "1" ]; then echo "Paso 5 omitido (EJECUTAR_ANALISIS=False)"; exit 0; fi
python -m analisis.estadistica resultados/final > /content/evidencia/analisis_final.txt 2>&1
cat /content/evidencia/analisis_final.txt
if [ -d resultados/suplementario_G300 ]; then
  python -m analisis.estadistica resultados/suplementario_G300 > /content/evidencia/analisis_suplementario_G300.txt 2>&1
  echo ">> también se regeneró el suplementario"
fi

## 6 · Guardar la evidencia
Copia `/content/evidencia/` a Drive y descarga un `.zip`.
Esa carpeta es la que se cita en la bitácora (versiones, pruebas, contraste con sboxU, reproducibilidad).

In [8]:
import shutil
from google.colab import files

!ls -la /content/evidencia
destino = f"{DRIVE_DIR}/evidencia_colab"
shutil.rmtree(destino, ignore_errors=True)
shutil.copytree("/content/evidencia", destino)
shutil.make_archive("/content/evidencia_colab", "zip", "/content/evidencia")
files.download("/content/evidencia_colab.zip")
print("Respaldo en Drive:", destino)

total 28
drwxr-xr-x 2 root root 4096 Nov 14 21:29 .
drwxr-xr-x 1 root root 4096 Nov 14 21:24 ..
-rw-r--r-- 1 root root   38 Nov 14 21:28 contraste_final_200.txt
-rw-r--r-- 1 root root   28 Nov 14 21:27 contraste_sboxu_51.txt
-rw-r--r-- 1 root root  343 Nov 14 21:24 entorno_colab.txt
-rw-r--r-- 1 root root  192 Nov 14 21:27 pytest.txt
-rw-r--r-- 1 root root   44 Nov 14 21:39 reproducibilidad.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Respaldo en Drive: /content/drive/MyDrive/Sbox-Ga/evidencia_colab




**Notas**
- Si Colab reinicia la sesión se pierden `/content/sage-env` y `/content/sbox-ga`: basta con volver a ejecutar desde el paso 1 (el paso 2 es el lento).
- No hace falta modificar ningún archivo del repo en Colab.